# Tokenizer
We started to experiment with `Dream of Red Chamber`

In [1]:
import requests
import pandas as pd
from pathlib import Path
import json
import regex
import time

In [2]:
pat = regex.compile(
    r"[\p{P}\p{S}]|'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+[\p{L}&&\n^\p{Han}]++|[\p{Han}]{1,2}+|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s++$|\s*[\r\n]|\s+(?!\S)|\s")

In [3]:
def get_locations(corpus):

    DATA = Path("../data")
    DATA.mkdir(exist_ok=True)
    TK_DATA = DATA / "tokenizer"
    TK_DATA.mkdir(exist_ok=True)
    CORPUS_DATA = TK_DATA / corpus
    CORPUS_DATA.mkdir(exist_ok=True)
    return CORPUS_DATA

In [5]:
CORPUS = "drc"
CORPUS_DATA = get_locations(CORPUS)

In [6]:
if CORPUS =="xyj":
    url = "https://raw.githubusercontent.com/tennessine/corpus/refs/heads/master/%E8%A5%BF%E6%B8%B8%E8%AE%B0.txt"

elif CORPUS == "wm":
    url = "https://github.com/tennessine/corpus/raw/refs/heads/master/%E6%B0%B4%E6%B5%92%E4%BC%A0.txt"

elif CORPUS == "rtk":
    url = "https://github.com/tennessine/corpus/raw/refs/heads/master/%E4%B8%89%E5%9B%BD%E6%BC%94%E4%B9%89.txt"

elif CORPUS == "drc":
    url = "https://github.com/tennessine/corpus/raw/refs/heads/master/%E7%BA%A2%E6%A5%BC%E6%A2%A6.txt"

In [7]:
def get_text(url):
    if (CORPUS_DATA / "corpus.txt").exists() is False:
        text = requests.get(url).text
        with open(CORPUS_DATA / "corpus.txt", "w") as f:
            f.write(text)
    else:
        text = (CORPUS_DATA / "corpus.txt").read_text()

    return text

In [8]:
text = get_text(url)

In [9]:
text[:100]

'《红楼梦》曹雪芹\n\n\n第一回  甄士隐梦幻识通灵\u3000贾雨村风尘怀闺秀\n\n    \t\t\n\n    此开卷第一回也．作者自云：因曾历过一番梦幻之后，故将真事隐去，而借"通灵"之说，撰此《石头记》一书也．故曰'

In [10]:
len(text)

855342

In [11]:
utf8 = list(text.encode())

In [12]:
len(utf8)

2517394

In [13]:
freq = dict()

for c1, c2 in zip(utf8,utf8[1:]):
    freq[(c1, c2)] = freq.get((c1, c2),0) + 1

In [14]:
most_freq = sorted(tuple((k, v ) for k,v in freq.items()), key=lambda x: -x[1])
most_freq[:5]

[((239, 188), 93292),
 ((188, 140), 57150),
 ((228, 184), 50516),
 ((228, 186), 42215),
 ((140, 229), 22965)]

In [15]:
# code_list = list(utf8)

code_collection = list(list(sub_text.encode("utf8")) for sub_text in pat.findall(text))

codes_start = 0
utf_len = len(list(text.encode("utf8")))
for l in code_collection:

    codes_start = max(max(l), codes_start)

vocab = dict()


def merge(ids, temp_vocab):
    if len(ids) < 2:
        return ids
    new_ids = []
    i = 0
    while i < len(ids):
        if i == len(ids) - 1:
            new_ids.append(ids[i])
            break
        token = temp_vocab.get((ids[i], ids[i + 1]))
        if token is not None:
            new_ids.append(token)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

logs = []

last_ts = time.time()
while True:
    freq = dict()

    for code_list in code_collection:

        if len(code_list) < 2:
            continue

        for c1, c2 in zip(code_list, code_list[1:]):
            freq[(c1, c2)] = freq.get((c1, c2), 0) + 1

    most_freq = sorted(tuple((k, v ) for k,v in freq.items()), key=lambda x: -x[1])

    temp_vocab = dict()
    top_freq_ids1 = set()
    top_freq_ids2 = set()

    for (c1, c2), ct in most_freq:
        if c1 in top_freq_ids2 or c2 in top_freq_ids1 or ct < 5:
            break
        top_freq_ids1.add(c1)
        top_freq_ids2.add(c2)

        codes_start += 1

        temp_vocab[(c1, c2)] = codes_start

        vocab[codes_start] = (c1, c2)

    # print(temp_vocab)

    if len(temp_vocab) == 0:
        print(f"\nFinished Vocab:{len(vocab)} /{ct} / {ct/utf_len:.5f}")
        break

    # print(f"Try to merge {len(temp_vocab)}")
    new_code = codes_start

    if len(vocab) % 100 == 0:
        print(f"V:{len(vocab)}x", end="\t")

    ct = 0

    for i in range(len(code_collection)):
        code_collection[i] = merge(code_collection[i], temp_vocab)
        ct += len(code_collection[i])

    new_ts = time.time()

    logs.append({
        "vocab_len": len(vocab),
        "code_list_len": ct,
        "compression": ct / utf_len,
        "time": new_ts - last_ts,
    })

    last_ts = new_ts
    

V:1200x	V:4500x	V:4900x	V:5100x	V:7900x	V:8700x	V:8900x	V:11500x	V:11800x	V:14100x	
Finished Vocab:15401 /4 / 0.00000


In [16]:
df = pd.DataFrame(logs)
df

,vocab_len,code_list_len,compression,time
0,1,2424102,0.962941,0.810769
1,4,2274221,0.903403,0.788748
2,13,2109326,0.837901,0.885633
3,24,1968207,0.781843,0.955178
4,34,1869686,0.742707,0.720292
...,...,...,...,...
803,15333,540449,0.214686,0.440558
804,15339,540419,0.214674,0.371547
805,15378,540224,0.214597,0.405768
806,15387,540179,0.214579,0.307947


In [17]:
with open(CORPUS_DATA / "vocab.json", "w") as f:
    f.write(json.dumps(vocab, indent=2), )

with open(CORPUS_DATA / "freq.json", "w") as f:
    f.write(json.dumps(most_freq, indent=2))

In [18]:
df = pd.DataFrame(logs)
df.to_csv(str(CORPUS_DATA / "logs.csv"), index=False)

## Retro Analysis

In [19]:
import json
import pandas as pd

In [20]:
df = pd.read_csv(str(CORPUS_DATA / "logs.csv"))

In [21]:
with open(CORPUS_DATA/"vocab.json") as f:
    vocab = dict((int(k), v) for k, v in json.loads(f.read()).items())

with open(CORPUS_DATA/"freq.json") as f:
    most_freq = json.loads(f.read())

In [22]:
from plotly import express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def plot_df(df):
    cols = ["vocab_len", "code_list_len", "compression", "time"]
    titles = ["Vocab Size", "Code List Length", "Compression Ratio", "Time per Step (s)"]

    fig = make_subplots(rows=2, cols=2, subplot_titles=titles)

    for i, (col, title) in enumerate(zip(cols, titles)):
        row, col_idx = divmod(i, 2)
        fig.add_trace(
            go.Scatter(x=df.index, y=df[col], mode="lines", name=title),
            row=row + 1, col=col_idx + 1,
        )

    fig.update_layout(height=600, title_text="BPE Tokenizer Training", showlegend=False)
    fig.show()


In [23]:
plot_df(df)

In [24]:
pair_to_code = dict((tuple(v), int(k)) for k, v in vocab.items())

In [25]:
from loguru import logger

In [26]:
def encode(text: str, pair_to_code):
    code_list = list(text.encode("utf8"))
    if len(code_list) < 2:
        return code_list
    while len(code_list) > 1:
        temp_vocab = dict()
        for c1, c2 in zip(code_list, code_list[1:]):
            token = pair_to_code.get((c1, c2))
            if token is None:
                continue
            else:
                temp_vocab[(c1, c2)] = token
        if len(temp_vocab) == 0:
            return code_list


        pair = min(temp_vocab, key=lambda x: temp_vocab[x])
        token = temp_vocab[pair]

        new_code_list = []

        i = 0
        while i < len(code_list):
            if i < len(code_list) - 1 and code_list[i] == pair[0] and code_list[i + 1] == pair[1]:
                logger.info(f"[REPLACE]{code_list[i]}, {code_list[i + 1]} -> {token}")
                new_code_list.append(token)
                i += 2
            else:
                new_code_list.append(code_list[i])
                i += 1
        
        code_list = new_code_list
            
    return code_list

In [30]:
list("宝钗".encode("utf8"))

[229, 174, 157, 233, 146, 151]

In [31]:
token_ids = encode("宝钗", pair_to_code=pair_to_code)
print(token_ids)

2026-06-27 00:44:23.855 | INFO     | __main__:encode:25 - [REPLACE]229, 174 -> 256
2026-06-27 00:44:23.858 | INFO     | __main__:encode:25 - [REPLACE]256, 157 -> 303
2026-06-27 00:44:23.862 | INFO     | __main__:encode:25 - [REPLACE]233, 146 -> 396
2026-06-27 00:44:23.868 | INFO     | __main__:encode:25 - [REPLACE]396, 151 -> 520
2026-06-27 00:44:23.873 | INFO     | __main__:encode:25 - [REPLACE]303, 520 -> 530


[530]


In [32]:
def decode(tokens, vocab):
    result = list(tokens)
    
    while True:
        i = 0
        new_result = []
        temp_vocab = dict((i, vocab[i]) for i in result if i in vocab)
        if len(temp_vocab) == 0:
            break
        token = max(temp_vocab, key=lambda x: x)
        c1, c2 = temp_vocab[token]
        for t in result:
            if t == token:
                new_result.append(c1)
                new_result.append(c2)
            else:
                new_result.append(t)
        result = new_result
        
    return bytes(result).decode(errors="replace")


def by_token_decode(tokens, vocab):
    results = []
    for token in tokens:
        by_token_str = decode([token], vocab)
        results.append(by_token_str)
    return results
    

In [33]:
for i in range(1000, 2000):
    print(
        decode([i], vocab=vocab),
        end="\n" if i % 30 == 0 else ", ")

被, 薛蟠, 传, 邢夫人, �, 恐, 欢, 位, 在那里, 及, 侍, 仍, 小丫头, �, 现, 受, 好了, 荣, 领, �, 姊
八, �, 穿, 怪, 办, 到了, �儿, 高, 象, 原来, 苦, 妙, 记, 文, 伤, 陪, 落, 进去, 芳, 爱, 认, 我也, �, �, 孩子, 吃了, 家里, 林黛玉, �, 叹
咐, 备, 不成, 光, 重, 姊妹, 麝, 村, 山, 齐, 骂, 去罢, 厮, 有一, 寻, 这会子, 人来, 跑, 到底, 很, 闻, 他的, 烟, 死了, 香菱, 即, 交, 环, 远, 遂
包, 好的, 疼, 芸, 混, 古, �, 上来, 麝月, 鬟, 宝玉笑道, 姨娘, �, 明白, 不在, 了一个, 赏, 叔, 略, 不知道, 连忙, 哥哥, 明儿, 静, 养, 丫鬟, 路, 忘, 别人, 祖
合, 直, 石, 心中, 灯, 舅, 贾蓉, 千, 甚, 看见, 活, 又是, 方才, 不用, 容, 立, 指, 杯, 彩, 说是, 里头, 景, 秦, 叫人, 马, 戏, �, 回去, 身上, 欲
怨, 料, 你的, 桂, �, 还是, 周瑞, �, 非, 全, 热, 赦, 贵, 起身, 小厮, 闷, 愿, 于是, 强, 了一回, 嫂, �, 谢, 流, 意思, 二奶奶, 转, 给他, 省, 果然
怎么样, 免, 乐, 我说, 那个, 已经, 跟前, 论, 紧, 不如, 旁, 总, 画, 只有, 恩, 主意, 极, 奴, 赖, 仙, 空, 刻, 万, 谁知, 伏, 越发, �, 唬, �, 赵
唤, 买, 瞧瞧, 有人, 便说, 惜春, 有什么, 士, 罪, 房中, 又说, 车, 女儿, 句话, 呆, 错, 教, �, 体, 失, 贾芸, 复, 一日, 洗, 完了, 凡, 奇, 过去, 商, 派
只怕, 琴, 钟, 贾赦, �了, 难道, 好些, 柳, 种, �, 独, 首, 鬼, 吩, 灵, 悲, 了几, 胡, 糊, 喜欢, 吩咐, 倒是, 帐, �, 就是了, �, 儿的, 雨村, 宁, 断
我就, 不住, 贾母道, 《, 场, 最, 议, 带了, 况且, 有了, 不�, 目, 窗, 兄弟, 嫂子, �, 的是, 男, �了, 此时, 席, 青, 系, 一声, 看着, 通, 添, 渐, 悄悄, 叫我
》, 海, 顾, 丢,

In [34]:
for i in range(12345, 13345):
    try:
        print(
            decode([i], vocab=vocab),
            end="\n" if i % 30 == 0 else "| ")
    except ValueError:
        print(i)

别的都| 露了| 无味| 玉�| 渚| 正殿| 朴| 想不起| 闸| 这叫作| 久了| 众人问| 花卉| 成的| 旧路| 几个小厮
尽行| 二门前| 宝玉已| 将黛玉| 把你的| 向里| 瞧我| 出房| 十二个女孩子| 竟未| 戏来| 又请| 色色| 八日| 请旨| 香珠| 每一| 一试| 世代| 之先| 天仙| 泪如雨下| 筵宴| 吃年| 作戏| 挂着一| 死是活| 不告诉| 大笑道| 不出来的
难了| 放下心来| 不是你| 看了一遍| 这些丫头们| 吃了罢| 回来又| 吃了一碗| 难道他| 你们看| 狐媚子| 赌气去了| 只见晴雯| 留的| 自向| 定例| 留下我| 倚势| 就剩| 若果然| 毛病儿| 为由| 泪痕满面| 知识| 再不许| 八人| 清晨| 头疼| 事小| 歇歇儿
是那里来的|  宝玉笑道| 不说了| 人家有| 求饶| 小耗| 没见世面| 我把你| 故典| 还说是| 只听宝玉| 那林黛玉| 听了一听| 忙要| 样的| 你不过是| 没看见| 护着| 狐狸| 担待| 叫老太太| 你只说| 烧的| 这老| 又见他| 天长| 得罪人| 你只顾| 和姑娘们| 簪环
要睡| 堆着| 谁呢| 赢了| 不过是些| 贾环听了| 耳内| 环兄弟| 教导他| 更甚| 性的| 不尊重| 坏心| 跟了来| 和别人| 你这么个| 咱们是| 也不犯| 你起来| 穿了衣服| 手巾| 珠子| 不是的| 这屋子| 伏侍老太太| 我也罢了| 是为什么| 冷清清| 兴趣| 赶了
烹茶| 写毕| 乱着| 俊的| 轻浮| 不管了| 压倒| 那贾琏| 袖内| 弯着腰| 罐| 别说他| 都是你们| 平儿听说|  话说贾琏| 想了半日| 虽不是| 蠲| 唤了| 交与他| 你们听听| 疼宝玉| 倒说我| 玩物| 既这样说| 让他们| 点了| 你过来| 相离| 来去
散时| 衣包| 手道| 问的| 分辩| 不是他| 见说| 立足| 自得| 我问你| 埃| 不过一时| 灯来| 回复了| 唤来| 也自| 也是要| 想了想| 回首| 算盘| 一响| 作此| 年年| 词句| 适才| 思索| 如同| 命将| 依次| 这些小
不知何事| 多多的| 松柏| 我管| 封锁| 还自| 他说什么| 也无碍| 不算外| 前番| 快乐|  秋| 左思| 左思右| 左思右想| 宝玉要| 花瓣| 化了| 连饭| 直竖| 一辈子的

Turn down the logging level a little bit

In [35]:
import sys
logger.remove()
logger.add(sys.stderr, level="WARNING")

1

In [39]:
token_ids = encode("悄悄说道", pair_to_code=pair_to_code)
by_token_decode(token_ids, vocab=vocab)

['悄悄说道']

In [40]:
decode(encode(text[:1000], pair_to_code), vocab) == text[:1000]

True